# Лекция 02. Последовательности и работа с файлами

От значений и функций переходим к структурам текста и обмену данными с внешним миром.

## Цели

После лекции вы сможете:

- работать с индексами, срезами и методами строк и списков;
- форматировать строки и явно преобразовывать текстовые данные;
- строить простые comprehensions;
- разбирать структурированные варианты через `match/case`;
- создавать пути через `pathlib.Path`;
- читать и записывать текст с явной кодировкой;
- использовать `with` для управления ресурсами;
- читать traceback и обрабатывать ожидаемые ошибки;
- читать и записывать CSV и JSON.

## Перед началом

Понадобятся условия, циклы и функции из занятия 1. Демонстрации используют файлы из каталога `lesson02/data`; запускайте Jupyter из корня репозитория. Если notebook открыт с другим рабочим каталогом, вспомогательная ячейка найдёт соседний каталог `data`.

## Строки и списки как последовательности

Последовательность хранит элементы в порядке. Индексы начинаются с нуля, отрицательные индексы отсчитываются с конца. `len(sequence)` возвращает длину. Срез `sequence[start:stop:step]` создаёт новую последовательность; правая граница не включается.

In [ ]:
text = "economics"
values = [10, 20, 30, 40, 50]

print(text[0], text[-1], text[1:4])
print(values[:3], values[::2], values[::-1])

## Строковые операции

Строка неизменяема: методы создают новое значение. Часто используются `.strip()`, `.lower()`, `.upper()`, `.replace()`, `.split()`, `.startswith()`, `.endswith()` и `.isdigit()`. Метод `separator.join(parts)` соединяет строки.

Кодировка связывает символы и байты. При чтении и записи текста кодировку нужно указывать явно; в курсе используем UTF-8.

In [ ]:
raw = "  Household   Income  "
parts = raw.lower().split()
slug = "-".join(parts)
print(parts)
print(slug)
print(raw)

## Форматирование строк

Строки можно соединять оператором `+` и повторять оператором `*`. Обычно значения удобнее вставлять через f-строку: `f"Прибыль: {profit}"`.

Спецификатор `.2f` выводит число с двумя знаками после десятичной точки. Форматирование меняет представление для человека, но не само числовое значение.

In [ ]:
currency = "RUB"
amount = 1234.5
print(currency + ": " + str(amount))
print(f"{currency}: {amount:.2f}")
print("-" * 20)

## Ввод — это тоже текст

`input()` останавливает программу и возвращает введённую пользователем **строку**. Даже если пользователь ввёл `100`, результат имеет тип `str`. Для вычислений строку явно преобразуют: `int(text)` или `float(text)`.

В автоматических заданиях не печатайте приглашение внутри `input`: система ожидает только требуемый ответ. Тот же принцип действует для файлов, CSV и JSON: внешний текст надо разобрать и проверить.

In [ ]:
# В интерактивном режиме снимите комментарии и введите число.
# income_text = input()
# income = int(income_text)
# print(income * 12)

# Для воспроизводимой демонстрации подставим ту же строку вручную.
income_text = "75000"
income = int(income_text)
print(income * 12)

## Явные преобразования

- `int("42")` создаёт целое число;
- `float("3.14")` создаёт дробное число;
- `str(42)` создаёт строку;
- `bool(value)` показывает логическую интерпретацию значения.

Преобразование должно иметь смысл. Например, `int("3.14")` завершится ошибкой, потому что строка не записана в формате целого числа.

In [ ]:
quantity_text = "12"
quantity = int(quantity_text)
print(quantity + 1)

print(str(quantity) + " единиц")
print(bool(0), bool(1), bool(""), bool("0"))

## Списки и comprehensions

Список изменяем: элементы можно заменить, добавить через `.append()` или удалить через `.pop()`. Списковое включение `[expression for item in source if condition]` удобно для одного простого преобразования или фильтра. Если логика содержит несколько ветвей и состояний, обычный цикл читается лучше.

In [ ]:
prices = [100, 250, 180, 400]
expensive = [price for price in prices if price >= 200]
discounted = [price * 0.9 for price in expensive]
print(expensive)
print(discounted)
print(prices)

## Структурное сопоставление `match/case`

`match` проверяет шаблоны `case` сверху вниз и выполняет первый подошедший. Шаблон может одновременно проверить длину последовательности, литеральные части и извлечь значения. `|` объединяет варианты, а `_` обрабатывает остаток.

К шаблону можно добавить **guard** — дополнительное условие после `if`: `case pattern if condition`. Сначала должен совпасть сам шаблон, и только затем вычисляется guard; извлечённые шаблоном имена уже доступны в условии. Если guard ложен, Python продолжает проверять следующие `case`.

In [ ]:
def parse_command(command: str) -> str:
    match command.split():
        case ["show", name]:
            return f"show:{name}"
        case ["save", name] if name.isidentifier():
            # Шаблон извлёк name, guard проверил допустимость имени.
            return f"save:{name}"
        case ["save", _]:
            # Сюда попадёт save с именем, не прошедшим guard.
            return "invalid-name"
        case ["quit"] | ["exit"]:
            return "stop"
        case _:
            return "unknown"


for command in ["save report", "save annual-report", "exit"]:
    print(command, "->", parse_command(command))

## Файл — внешний ресурс

Переменные исчезают после завершения процесса, файл сохраняет данные. Работа с файлом включает путь, режим открытия, кодировку, чтение или запись и обязательное закрытие. Ошибки файловой системы являются нормальной частью интерфейса: файла может не быть, путь может указывать не туда, доступ может быть запрещён.

## Пути через `pathlib.Path`

`Path` представляет путь как объект и не заставляет вручную склеивать строки с `/` или `\`. Оператор `/` добавляет часть пути. Полезны свойства `.name`, `.suffix`, `.parent` и методы `.exists()`, `.is_file()`, `.read_text()`, `.write_text()`.

Относительный путь разрешается относительно текущего рабочего каталога процесса, а не обязательно относительно файла с кодом.

In [ ]:
from pathlib import Path

data_dir = Path("lesson02/data")
if not data_dir.exists():
    data_dir = Path("data")

numbers_path = data_dir / "numbers.txt"
print(numbers_path, numbers_path.name, numbers_path.suffix)
print(numbers_path.exists(), numbers_path.is_file())

## Чтение текста

`Path.read_text(encoding="utf-8")` удобно для небольшого файла, который помещается в память. `text.splitlines()` делит содержимое на строки. Для потокового чтения открывают файл и перебирают его построчно; эта модель станет особенно важна при изучении итераторов и генераторов.

In [ ]:
content = numbers_path.read_text(encoding="utf-8")
numbers = [int(line) for line in content.splitlines() if line.strip()]
print(repr(content))
print(numbers)

## Контекстный менеджер `with`

`with open(...) as file:` гарантирует закрытие файла и при успешном выполнении, и при исключении. Режимы: `r` — чтение, `w` — перезапись, `a` — добавление, `b` — двоичные данные. Для текста указывают `encoding`.

Не перехватывайте ошибку только ради закрытия файла: управление ресурсом уже решает `with`.

In [ ]:
parsed = []
with numbers_path.open(encoding="utf-8") as source:
    for line in source:
        stripped = line.strip()
        if stripped:
            parsed.append(int(stripped))

print(parsed)

## Запись без загрязнения рабочего каталога

Режим `w` заменяет существующее содержимое. Перед записью нужно понимать, какой путь будет изменён. В демонстрациях используем временный каталог, который удаляется автоматически. Для настоящего результата путь должен передаваться функции явно.

In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    report_path = Path(directory) / "report.txt"
    report_path.write_text("total=40\ncount=4\n", encoding="utf-8")
    print(report_path.read_text(encoding="utf-8"))

## Как читать traceback

Последняя строка traceback сообщает тип и причину ошибки, строки выше — место возникновения. Частые случаи:

- `SyntaxError` — нарушен синтаксис, например пропущено двоеточие;
- `NameError` — имя ещё не создано или написано иначе;
- `TypeError` — операция не поддерживает такие типы;
- `ValueError` — тип преобразования подходит, но содержимое строки имеет неверный формат;
- `FileNotFoundError` — файл не найден по вычисленному пути.

Не исправляйте ошибку наугад: сначала прочитайте её тип, строку и значения участвующих переменных.

Попробуйте по очереди выполнить выражения в отдельной пустой ячейке и прочитать последнюю строку сообщения:

```python
"100" + 20       # TypeError: складываются несовместимые типы
int("12.5")       # ValueError: строка не записана как целое число
unknown_name + 1   # NameError: имя не определено
Path("missing.txt").read_text()  # FileNotFoundError
```

После наблюдения удалите ошибочную ячейку или перезапустите kernel. Следующий шаг — перехватить только ту ошибку, которую программа действительно умеет обработать.

## Явная обработка ошибок

Перехватывайте только ошибку, которую можете содержательно обработать. `FileNotFoundError` позволяет сообщить об отсутствующем входе, `ValueError` — о неверном формате значения. Общий `except Exception` скрывает программные ошибки и затрудняет диагностику.

После `except` можно либо вернуть предусмотренный результат, либо выбросить новое исключение с предметным сообщением через `raise ... from error`.

In [ ]:
def read_one_integer(path: Path) -> int:
    try:
        text = path.read_text(encoding="utf-8").strip()
        return int(text)
    except FileNotFoundError:
        raise ValueError(f"Файл не найден: {path}")
    except ValueError as error:
        raise ValueError(f"В {path} записано не одно целое число") from error


with TemporaryDirectory() as directory:
    sample = Path(directory) / "one.txt"
    sample.write_text("42", encoding="utf-8")
    print(read_one_integer(sample))

## CSV

CSV хранит таблицу строками и столбцами, но правила кавычек и разделителей сложнее простого `.split(",")`. Используйте стандартный модуль `csv`. `csv.reader` возвращает каждую строку как список строк; преобразование чисел выполняется отдельно. `newline=""` рекомендуется при открытии CSV.

In [ ]:
import csv

csv_path = data_dir / "observations.csv"
with csv_path.open(encoding="utf-8", newline="") as source:
    reader = csv.reader(source)
    header = next(reader)
    rows = [row for row in reader]

print(header)
print(rows)

## JSON

JSON хранит числа, строки, логические значения, `null`, массивы и объекты. `json.load(file)` читает из открытого файла, `json.loads(text)` — из строки; `dump` и `dumps` выполняют обратное преобразование. JSON-объект становится словарём Python, а массив — списком. Подробная модель словарей рассматривается на занятии 5.

In [ ]:
import json

config_path = data_dir / "config.json"
with config_path.open(encoding="utf-8") as source:
    config = json.load(source)

print(config["title"])
print(config["precision"])

In [ ]:
records = [["cash", 100], ["deposit", -25]]

with TemporaryDirectory() as directory:
    target = Path(directory) / "transactions.json"
    with target.open("w", encoding="utf-8") as output:
        json.dump(records, output, ensure_ascii=False, indent=2)
    print(target.read_text(encoding="utf-8"))

## Граница между ядром и I/O

Функцию вычисления удобно отделять от чтения и записи. Тогда предметное ядро принимает значения Python и возвращает значения Python, а тонкий слой I/O отвечает за пути, кодировки и форматы. Такая граница упрощает тестирование и станет обязательной для групповых проектов.

In [ ]:
def positive_total(values: list[int]) -> int:
    total = 0
    for value in values:
        if value > 0:
            total += value
    return total


values = [int(line) for line in numbers_path.read_text(encoding="utf-8").splitlines() if line]
result = positive_total(values)
print(result)

## Типичные ошибки

- рассчитывать относительный путь от расположения исходного файла, а не от рабочего каталога;
- полагаться на системную кодировку;
- читать большой файл целиком без необходимости;
- разбирать CSV через `.split(",")`;
- открывать JSON как Python-код;
- забывать режим `w` или случайно перезаписывать входной файл;
- ловить `Exception` и продолжать работу с неполными данными;
- смешивать предметные вычисления с чтением и печатью.

## Неожиданно, но по правилам

Сначала предскажите результат, затем выполните ячейку.

1. Индекс должен указывать на существующий элемент, а границы среза могут выходить далеко за последовательность: Python просто вернёт доступную часть. Поэтому `text[100]` вызывает `IndexError`, а `text[100:]` возвращает пустую строку.
2. Запятая внутри кавычек в CSV относится к значению поля. Обычный `split(",")` не знает правил CSV и создаёт лишние части.
3. В JSON ключи объектов — строки. Числовой ключ словаря Python после записи и чтения JSON становится строковым.

In [ ]:
import csv
import json

text = "economics"
print(repr(text[100:]))
print(text[-100:3])
# print(text[100])  # IndexError: такого элемента нет.

line = '42,"food, drinks",100'
print(line.split(","))
print(next(csv.reader([line])))

encoded = json.dumps({1: "one"})
decoded = json.loads(encoded)
print(encoded, decoded, type(next(iter(decoded))).__name__)

## Самопроверка

1. Почему правая граница среза не включается?
2. Чем форматирование числа отличается от изменения его значения?
3. Почему `input()` требует явного преобразования для вычислений?
4. Когда comprehension лучше обычного цикла?
5. Чем `match/case` отличается от цепочки сравнений?
6. Когда вычисляется guard и что происходит, если он ложен?
7. Относительно чего разрешается относительный путь?
8. Зачем указывать `encoding="utf-8"`?
9. Что гарантирует `with`?
10. С какой части traceback следует начинать чтение?
11. Почему CSV нельзя надёжно разбирать через `split`?
12. Во что превращаются JSON-массив и JSON-объект?
13. Какие исключения следует перехватывать?
14. Зачем отделять предметное ядро от I/O?

## Итоги

- Строки и списки поддерживают индексы, срезы и обход.
- Внешний ввод приходит как текст: его форматируют для вывода и явно преобразуют для вычислений.
- `match/case` разбирает структурированные варианты, а guard уточняет совпавший шаблон дополнительным условием.
- `Path` делает работу с путями явной и переносимой.
- Текст читается и записывается с указанной кодировкой.
- `with` управляет временем жизни файла.
- Чтение ошибки начинается с последней строки traceback; перехватывают только ожидаемые исключения.
- CSV и JSON нужно обрабатывать стандартными модулями.
- I/O отделяют от предметного ядра.

На [семинаре](seminar.ipynb) эти элементы объединяются в небольшой конвейер преобразования данных.